# 분기하며 Tool 을 사용하는 랭그래프

In [ ]:
# 1. 랭체인 설치
%pip install langchain

# 2. 랭체인에서 OpenAI를 사용하기 위한 패키지
%pip install langchain_openai

# 3. 랭그래프 설치
%pip install langgraph

# 2. 랭체인에서 OpenAI를 사용하기 위한 패키지
%pip install langchain_openai

# 3. 랭그래프 설치
%pip install langgraph

In [1]:
import json
import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
MODEL = ChatOpenAI(model="gpt-4o-mini")

c:\Users\sycsn\OneDrive\STUDY\Coding\PYTHON\PY_LLM\MYVENV\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
class State(TypedDict):
    user_input: str
    messages: Annotated[list, add_messages]

### TOOL 정의
실제로 일을 수행하는 TOOL 을 정의 한다.
 - LLM 이 '특정TOOL을 사용하겠다는 판단'을 하고 Tool Call 만 한다.
 - 실제 실행은 뒤에서 만들 BasicToolNode 가 담당
 - LLM 의 Tool 사용 판단 기준
      1. 설명(docstring)
      2. 함수 이름
      3. 파라미터 이름과 타입과 순서

### 바인딩할 TOOL 목록
모델이 tool call 대상으로 인식할 수 있는 전체 TOOL LIST 생성

### bind_tools : 모델에 TOOL 바인딩
모델에게 "이런 도구들이 있다"라고 알려주는 과정
이 설정을 해야 모델이 tool_calls를 포함한 응답을 생성할 수 있다.

### 시스템 프롬프트 : Tool 사용 시점 지시

### BasicToolNode
AIMessage 에서 TOOL 호출 요청이 있을 때 이를 실행시키는 역할

# __init__ : tools by name : key[tool 이름], value[실제 tool 객체] 로 하는 dict. 를 생성하는 초기화 단계
{
    "add_numbers": <tool object>,
    "multiply_numbers": <tool object>,
    "reverse_text": <tool object>,
}
이렇게 저장해두면, 모델이 "add_numbers"를 호출하라고 했을 때
당 함수를 빠르게 찾아 실행할 수 있다.

# __call__ : 호출 메서드 - 클래스를 함수처럼 사용하게 해준다.
클래스를 Langgraph 노드처럼 동작하게 한다.
- Tool 사용 요청의 경우 마지막 메세지의 형식은
``` PYTHON
AIMessage(..., tool_calls=[...])
last_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "add_numbers",
            "args": {"a": 17, "b": 25},
            "id": "call_abc123",
            "type": "tool_call",
        },
        {
            "name": "multiply_numbers",
            ...
        }
    ],
)

for tool_call in last_message.tool_calls:
```

# Tool id
모델이 만든 각 도구 호출의 고유 식별자
왜 필요하냐면, 나중에 tool 실행 결과를 다시 모델에게 줄 때 어느 호출에 대한 결과인지 연결해야 하기 때문입니다.

 - 모델이 "add_numbers" 호출을 하나 만듦
 - 그 호출에 id를 붙임
 - 네 코드가 tool 실행
 - 결과를 ToolMessage(tool_call_id=...)로 돌려줌
 - 모델은 "아, 이 결과는 내가 아까 만든 그 호출의 결과구나" 하고 매칭함
* 특히 여러 개 tool call이 있을 때 중요합니다.

# ToolMessage
사람이나 AI가 말한 메시지가 아니라, 도구 실행 결과를 모델에게 돌려주는 메시지
ToolMessage는 도구 실행 결과를 대화 흐름에 끼워 넣는 특수 메시지
보통의 대화 흐름 : HumanMessage -> AIMessage(tool_calls) -> ToolMessage -> AIMessage(final answer)


In [3]:
# Tool 객체로 함수 정의
@tool
def add_numbers(number_a: int, number_b: int) -> str:
    # docstring
    """
    두 정수를 입력받아 합을 계산합니다.

    사용 시점:
        - 사용자가 두 수의 합이나 덧셈 결과를 요청할 때 사용합니다.

    입력값:
        - number_a: 첫 번째 정수
        - number_b: 두 번째 정수

    반환값:
        - "a + b = result" 형식의 문자열
    """

    return f"{number_a} + {number_b} = {number_a + number_b}"

@tool
def multiply_numbers(number_a: int, number_b:int) -> str:
    """
    두 정수를 곱한 결과를 반환한다.
    사용시점 : 사용자가 두수의 곱을 요청할 때 사용
    입력값: number_a, nubmer_b : 첫,두번째 정수
    반환값: "{a} x {b} = {a * b}" 형식의 문자열
    """

    return f"{number_a} x {number_b} = {number_a * number_b}"

@tool
def reverse_text(text: str) -> str:
    """
    입력 문자열을 뒤집어서 반환한다.
    """

    return text[::-1]

# TOOLS: tool 객체들 list
MY_TOOLS = [
    add_numbers,
    multiply_numbers,
    reverse_text
]

# TOOLS 바인딩
MODEL = MODEL.bind_tools(MY_TOOLS)

# 시스템 프롬프트 : TOOL 사용 시점을 알려줄 지시
SYSTEM_PROMPT = """
너는 도우미다.

규칙:
1. 여러 단계가 필요한 요청이라면 한 번에 하나의 도구만 호출하라.
2. 절대로 한 응답에서 두 개 이상의 도구를 동시에 호출하지 마라.
3. 어떤 도구의 출력값이 다음 단계의 입력이 되는 경우, 반드시 이전 도구 결과를 받은 뒤 다음 도구를 호출하라.
4. 계산이 필요하면 먼저 계산 도구를 호출하고, 그 결과를 확인한 다음에만 후속 도구를 호출하라.
5. 모든 도구 호출이 끝난 뒤에만 최종 답변을 한국어 한두 문장으로 정리하라.
""".strip()

In [ ]:
# NODE_run_tools
class NODE_run_tools:
    """마지막 AIMessage의 tool_calls를 읽어서 실제 tool을 실행하는 노드"""

    # 전체 Tools Dict 생성 - tools_by_name ; 인자로 MY_TOOLS 를 받는다.
    def __init__(self, tools:list) -> None:
        self.tools_by_name = {
            tool.name: tool for tool in tools
        }

    # 노드처럼 동작하는 Call 호출 메서드
    def __call__(self, state: dict):
        # 1. messages 읽기
        messages = state.get("messages", [])
        last_message = messages[-1] #AIMessage(... tool_calls=[...])

        # 2. 모델이 요청한 Tool 순회
        ToolResults = []
        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"] # "add_numbers"
            tool_args = tool_call["args"] # {"a": 17, "b": 25}
            tool_id   = tool_call["id"]   # 

            # 3. 실제 Tool 함수 실행
            tool_result = self.tools_by_name[tool_name].invoke(tool_args)

            ToolResults.append(
                ToolMessage(
                    content = json.dumps(tool_result, ensure_ascii=False), # 실제 Tool 사용 결과가 ToolMessage의 content, Tool 함수 return 값을 문자열로 변환하여 LLM 에게 전달하기 위해, json.dumps 사용하는 편이다.
                    name = tool_name,      # 요청받아 실행했던 TOOL 이름 붙여서 반환
                    tool_call_id = tool_id # AI 가 Tool 요청 때 지정한 ID를 붙여서 반환해준다. 식별.
                )
            )
            # ToolResults = [ToolMessage(...),ToolMessage(...),...]
        
        # 4. ToolMessage : Tool 실행 결과들을 ToolMessage 들이 담긴 리스트로 추가한다. 
        return {"messages": ToolResults} 

### NODE
1. init_system : 첫 턴에만 SystemMessage를 State 에 추가, 이후 턴에는 중복 삽입을 막음
2. add_user_message : 이번 턴의 user_input을 HumanMessage로 변환해 추가
3. generate : 준비된 state["messages"]만 그대로 invoke(...)

### 대화 턴 진행에 따른 State["messages"]
[
    SystemMessage(BASE), #1st Turn
    HumanMessage(...),   #1st
    AIMessage(...),      #1st

    HumanMessage(...),   #2nd
    AIMessage(...),      #2nd

    HumanMessage(...),   #3rd
    AIMessage(...),      #3rd


In [5]:
def NODE_init_system(state: State):
    """첫 턴에만 SystemMessage를 messages 맨 앞에 추가한다."""
    messages = state.get("messages", []) #massages 없으면 빈 리스트 get

    # 메세지가 이미 있고, 첫 메세지가 시스템 메세지 객체면 Node 스킵
    if messages and isinstance(messages[0], SystemMessage):
        return {}
    
    else:
        return {
            "messages" : [
                SystemMessage(content=SYSTEM_PROMPT)
            ]
        }
    
def NODE_add_user_message(state: State):
    """이번 턴의 사용자 입력을 HumanMessage로 변환해 messages에 추가한다."""
    
    return {
        "messages" : [
            HumanMessage(content = state["user_input"])
        ]
    }

def NODE_generate(state: State):
    """현재 상태(messages)를 바탕으로 다음 AIMessage를 생성한다."""
    
    response = MODEL.invoke(state["messages"])
    return {"messages": [response]}

### 라우팅 함수
조건 분기 함수
generate 노드의 출력(AIMessage)를 보고,
이를 기준삼아 어디로 이동할지 결정한다.

 * 함수의 반환값은 노드 이름 자체. e.g., START,END,"generate"
 * 노드가 아니므로, add_node 하지 않고 add_conditional_edges()에서 바로 사용한다.

### add_conditional_edges
graph_builder.add_conditional_edges(
    source : 시작노드이름,
    path : 분기판단함수,
    path_map : 라우트 반환값 - 노드 매핑
)

```PYTHON
graph_builder.add_conditional_edges(
    "generate",  # generate 실행이 끝난 뒤 분기 판단을 함
    route_tools,
    {"run_tools": "run_tools", END: END},
)
```
path_map이 “반환값”과 “실제 노드명”을 분리해주는 층

# hasattr
Has Attribute : hasattr()는 어떤 객체가 특정 속성(attribute)을 가지고 있는지 확인하는 함수

In [6]:
def ROUTE_run_tools(state: State):
    messages = state.get("messages", [])
    if not messages:
        return END
    
    last_message = messages[-1]
    
    # 조건 분기
    # 1. 마지막 메시지가 tool call을 포함하고 있는 경우
    if isinstance(last_message, AIMessage) and last_message.tool_calls :
        return "GOTO run_tools"
    # 2. tool_calls가 없다면 (최종 답변인 경우)
    else:
        return "DONE"

### 그래프 조립
# 1. 빌더 생성
graph_builder = StateGraph(State)
"이 그래프는 State 구조를 공유 상태로 사용합니다"

# 2. 노드 등록
START -> init_system -> generate ---Route---> (run_tools -> generate)*N -> END

# 3. 노드 연결

# 4. 빌더 컴파일

In [7]:
def MY_build_graph(checkpointer=None):
    # 1. 빌더 생성
    graph_builder = StateGraph(State)

    # 2. 노드 등록
    graph_builder.add_node("init_system", NODE_init_system)
    graph_builder.add_node("add_user_message", NODE_add_user_message)
    graph_builder.add_node("generate", NODE_generate)
    graph_builder.add_node("run_tools", NODE_run_tools(MY_TOOLS)) # Tool 실행 노드 (class, MY_TOOLS 인자 전달)

    # 3. 노드 연결
    graph_builder.add_edge(START, "init_system")
    graph_builder.add_edge("init_system", "add_user_message")
    graph_builder.add_edge("add_user_message", "generate")
    # generate -> (run_tools / END)
    graph_builder.add_conditional_edges(
        "generate",        # 출발 NODE
        ROUTE_run_tools,   # 조건 분기 라우트 함수
        {"GOTO run_tools": "run_tools", "DONE": END} # 도착지: 라우트 함수 반환값 매칭 딕셔너리
    )
    # tool 실행 후 다시 generate : 결과에 대한 최종 자연어 답변 생성
    graph_builder.add_edge("run_tools", "generate")

    # 4. 빌더 컴파일
    MY_graph = graph_builder.compile(checkpointer=checkpointer)

    return MY_graph

In [16]:
# 디버깅 함수 01. 최종 답변 확인
def ask_graph(question: str) -> str:
    graph = MY_build_graph()
    
    result = graph.invoke({
        "user_input" : question,
        "messages" : []
    })

    final_message = result["messages"][-1]
    print(f"[FINAL_ANSWER] : {final_message.content}")

    return final_message.content

# 디버깅 함수 02. Stream 방식 실행 - 각 노드 단계별 State 출력
def stream_graph(question: str):
    graph = MY_build_graph()

    for event in graph.stream(
        {"user_input" : question, "messages" : []},
        stream_mode='values',
    ):
        messages = event.get("messages", [])

        number_message = len(messages)
        if number_message == 0:
            last_message = None
            type_message = 'EMPTY'
            last_message_content = ""
        else:
            last_message = messages[-1]
            type_message = type(last_message).__name__
            last_message_content = last_message.content

        # 메세지 출력
        print(f"[#{number_message}][{type_message}] : {last_message_content} ")

        # AI Tool Call 정보 for NODE_run_tool
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            print("     ■ [AI's TOOL CALLS]")
            for idx, tool_call in enumerate(last_message.tool_calls):
                print(f"        □ TOOL #{idx}")
                print(f"            □ NAME(input) : {tool_call['name']}({tool_call['args']})")
                print(f"            □ ID          : {tool_call['id']}")
        
        # ToolMessage 정보 for NODE_generate
        if last_message is not None and isinstance(last_message, ToolMessage):
            print(f"     ■ [TOOL Result]")
            print(f"        □ NAME   : {last_message.name}")
            print(f"        □ output : {last_message.content}")
            print(f"        □ ID     : {last_message.tool_call_id}")


In [17]:
i_said = input("YOU : ")
input("====== Debug #1; Final Answer =====\n")
ask_graph(i_said)
input("====== Debug #2; Stream State =====\n")
stream_graph(i_said)

#3과 7을 합하는 것에 대해 말해. 그리고 그걸 뒤집어. 그리고 그걸 다시 뒤집어.

[FINAL_ANSWER] : 3과 7을 합하면 10이 됩니다. 이를 뒤집으면 "01 = 7 + 3"이 되고, 다시 뒤집으면 원래 문장인 "3 + 7 = 10"으로 돌아옵니다.
[#0][EMPTY] :  
[#1][SystemMessage] : 너는 도우미다.

규칙:
1. 여러 단계가 필요한 요청이라면 한 번에 하나의 도구만 호출하라.
2. 절대로 한 응답에서 두 개 이상의 도구를 동시에 호출하지 마라.
3. 어떤 도구의 출력값이 다음 단계의 입력이 되는 경우, 반드시 이전 도구 결과를 받은 뒤 다음 도구를 호출하라.
4. 계산이 필요하면 먼저 계산 도구를 호출하고, 그 결과를 확인한 다음에만 후속 도구를 호출하라.
5. 모든 도구 호출이 끝난 뒤에만 최종 답변을 한국어 한두 문장으로 정리하라. 
[#2][HumanMessage] : 3과 7을 합하는 것에 대해 말해. 그리고 그걸 뒤집어. 그리고 그걸 다시 뒤집어. 
[#3][AIMessage] :  
     ■ [AI's TOOL CALLS]
        □ TOOL #0
            □ NAME(input) : add_numbers({'number_a': 3, 'number_b': 7})
            □ ID          : call_8IRwvCFqZsNf6b2PgAlRLdYu
[#4][ToolMessage] : "3 + 7 = 10" 
     ■ [TOOL Result]
        □ NAME   : add_numbers
        □ output : "3 + 7 = 10"
        □ ID     : call_8IRwvCFqZsNf6b2PgAlRLdYu
[#5][AIMessage] :  
     ■ [AI's TOOL CALLS]
        □ TOOL #0
            □ NAME(input) : reverse_text({'text': '3 + 7 = 10'})
            □ ID          : call_0juzZjF32O6geZ

[FINAL_ANSWER] : 3과 7을 합하면 10이 됩니다. 이를 뒤집으면 "01 = 7 + 3"이 되고, 다시 뒤집으면 원래 문장인 "3 + 7 = 10"으로 돌아옵니다.


[#0][EMPTY] :  
[#1][SystemMessage] : 너는 도우미다.

규칙:
1. 여러 단계가 필요한 요청이라면 한 번에 하나의 도구만 호출하라.
2. 절대로 한 응답에서 두 개 이상의 도구를 동시에 호출하지 마라.
3. 어떤 도구의 출력값이 다음 단계의 입력이 되는 경우, 반드시 이전 도구 결과를 받은 뒤 다음 도구를 호출하라.
4. 계산이 필요하면 먼저 계산 도구를 호출하고, 그 결과를 확인한 다음에만 후속 도구를 호출하라.
5. 모든 도구 호출이 끝난 뒤에만 최종 답변을 한국어 한두 문장으로 정리하라. 
[#2][HumanMessage] : 3과 7을 합하는 것에 대해 말해. 그리고 그걸 뒤집어. 그리고 그걸 다시 뒤집어. 
[#3][AIMessage] :  
     ■ [AI's TOOL CALLS]
        □ TOOL #0
            □ NAME(input) : add_numbers({'number_a': 3, 'number_b': 7})
            □ ID          : call_8IRwvCFqZsNf6b2PgAlRLdYu
[#4][ToolMessage] : "3 + 7 = 10" 
     ■ [TOOL Result]
        □ NAME   : add_numbers
        □ output : "3 + 7 = 10"
        □ ID     : call_8IRwvCFqZsNf6b2PgAlRLdYu
[#5][AIMessage] :  
     ■ [AI's TOOL CALLS]
        □ TOOL #0
            □ NAME(input) : reverse_text({'text': '3 + 7 = 10'})
            □ ID          : call_0juzZjF32O6geZ17bhEJVk3J
[#6][ToolMessage] : "01 = 7 + 3" 
     ■ [TOOL Result]
        □ NAME   : reverse_text
        □ output : "01 = 7 + 3"
        □ ID     : call_0juzZjF32O6geZ17bhEJVk3J
[#7][AIMessage] :  
     ■ [AI's TOOL CALLS]
        □ TOOL #0
            □ NAME(input) : reverse_text({'text': '01 = 7 + 3'})
            □ ID          : call_AJuoq1p9xYdOw8qq2EYl9Nmp
[#8][ToolMessage] : "3 + 7 = 10" 
     ■ [TOOL Result]
        □ NAME   : reverse_text
        □ output : "3 + 7 = 10"
        □ ID     : call_AJuoq1p9xYdOw8qq2EYl9Nmp
[#9][AIMessage] : 3과 7을 합하면 10이 됩니다. 이를 뒤집으면 "01 = 7 + 3"이 되고, 다시 뒤집으면 원래대로 "3 + 7 = 10"이 됩니다.

In [ ]:
def run_memory_chatbot(memory: MemorySaver, thread_id: str):
    """MemorySaver를 이용해 대화가 누적되는 터미널 챗봇을 실행한다."""

    graph = MY_build_graph(checkpointer = memory)

    # thread_id가 같으면 같은 대화 스레드로 간주되어 상태가 이어진다.
    config_now = {"configurable": {"thread_id": thread_id}}

    print("대화형 챗봇을 시작합니다. 종료하려면 exit 또는 quit 를 입력하세요.")

    while True:
        user_input = input("[User] : ").strip()
        if user_input.lower() in {'q','ㅃ'}:
            print("챗봇을 종료합니다.")
            break

        for event in graph.stream(
            {"user_input": user_input, "messages": []},
            config=config_now,
            stream_mode="values",
        ):
            messages = event.get('messages',[])
            if not messages:
                continue
            else:
                messages[-1].pretty_print()
                print(f"    (현재 메시지 개수: {len(event['messages'])})\n\n")


In [30]:
MY_memory = MemorySaver() 
MY_thread_id = "ANONY-123"
run_memory_chatbot(MY_memory, MY_thread_id)

# 기존 Memory 사용시 이전 챗봇 사용 이력 이어서 가능 
MY_thread_id = "ANONY-123"
run_memory_chatbot(MY_memory, MY_thread_id)

대화형 챗봇을 시작합니다. 종료하려면 exit 또는 quit 를 입력하세요.
================================== Ai Message ==================================

감사합니다! ㅋㅋ 더 필요하신 것이 있으면 말씀해 주세요!
현재 메시지 개수: 17
================================ Human Message =================================

아까 뭐했지?
현재 메시지 개수: 18
================================== Ai Message ==================================

우리는 사과 317개와 배 215개의 총합이 532개란 것을 계산했어요. 그 후, 그 결과를 거꾸로 말해달라고 하셔서, 결과를 두 번 거꾸로 표현하는 대화도 나누었습니다.
현재 메시지 개수: 19
================================== Ai Message ==================================

우리는 사과 317개와 배 215개의 총합이 532개란 것을 계산했어요. 그 후, 그 결과를 거꾸로 말해달라고 하셔서, 결과를 두 번 거꾸로 표현하는 대화도 나누었습니다.
현재 메시지 개수: 19
================================ Human Message =================================

ㅂ
현재 메시지 개수: 20
================================== Ai Message ==================================

무엇을 도와드릴까요? 궁금한 점이 있으면 말씀해 주세요!
현재 메시지 개수: 21
챗봇을 종료합니다.


대화형 챗봇을 시작합니다. 종료하려면 exit 또는 quit 를 입력하세요.
=============================== [System Message] ================================

너는 도우미다.

규칙:
1. 여러 단계가 필요한 요청이라면 한 번에 하나의 도구만 호출하라.
2. 절대로 한 응답에서 두 개 이상의 도구를 동시에 호출하지 마라.
3. 어떤 도구의 출력값이 다음 단계의 입력이 되는 경우, 반드시 이전 도구 결과를 받은 뒤 다음 도구를 호출하라.
4. 계산이 필요하면 먼저 계산 도구를 호출하고, 그 결과를 확인한 다음에만 후속 도구를 호출하라.
5. 모든 도구 호출이 끝난 뒤에만 최종 답변을 한국어 한두 문장으로 정리하라.
현재 메시지 개수: 1
=============================== [Human Message] =================================

안녕
현재 메시지 개수: 2
=============================== [Ai Message] ==================================

안녕하세요! 무엇을 도와드릴까요?
현재 메시지 개수: 3
=============================== [Ai Message] ==================================

안녕하세요! 무엇을 도와드릴까요?
현재 메시지 개수: 3
=============================== [Human Message] =================================

사과 317개와 배 215개가 있으면 과일은 총 몇개야?
현재 메시지 개수: 4
=============================== [Ai Message] ==================================
Tool Calls:
  add_numbers (call_DNL6ldOLMpUZ27NJdFdexPOe)
 Call ID: call_DNL6ldOLMpUZ27NJdFdexPOe
  Args:
    number_a: 317
    number_b: 215
현재 메시지 개수: 5
=============================== [Tool Message] =================================
Name: add_numbers

"317 + 215 = 532"
현재 메시지 개수: 6
=============================== [Ai Message] ==================================

사과 317개와 배 215개를 합치면 총 532개의 과일이 있습니다.
현재 메시지 개수: 7
=============================== [Ai Message] ==================================

사과 317개와 배 215개를 합치면 총 532개의 과일이 있습니다.
현재 메시지 개수: 7
=============================== [Human Message] =================================

너의 대답을 거꾸로 말해봐
현재 메시지 개수: 8
=============================== [Ai Message] ==================================
Tool Calls:
  reverse_text (call_S9YJqJlteRO0L6dZ9ELGQIYp)
 Call ID: call_S9YJqJlteRO0L6dZ9ELGQIYp
  Args:
    text: 532개의 과일이 있습니다.
현재 메시지 개수: 9
=============================== [Tool Message] =================================
Name: reverse_text

".다니습있 이일과 의개235"
현재 메시지 개수: 10
=============================== [Ai Message] ==================================

"532개의 과일이 있습니다."를 거꾸로 말하면 ".다니습있 이일과 의개235"입니다.
현재 메시지 개수: 11
=============================== [Ai Message] ==================================

"532개의 과일이 있습니다."를 거꾸로 말하면 ".다니습있 이일과 의개235"입니다.
현재 메시지 개수: 11
=============================== [Human Message] =================================

다시 거꾸로 말해봐 ㅋㅋ
현재 메시지 개수: 12
=============================== [Ai Message] ==================================
Tool Calls:
  reverse_text (call_QBLMxoxOaQMZ34qvQFQoC8M1)
 Call ID: call_QBLMxoxOaQMZ34qvQFQoC8M1
  Args:
    text: .다니습있 이일과 의개235
현재 메시지 개수: 13
=============================== [Tool Message] =================================
Name: reverse_text

"532개의 과일이 있습니다."
현재 메시지 개수: 14
=============================== [Ai Message] ==================================

거꾸로 말한 결과를 다시 거꾸로 말하면 "532개의 과일이 있습니다."입니다. ㅋㅋ
현재 메시지 개수: 15
=============================== [Ai Message] ==================================

거꾸로 말한 결과를 다시 거꾸로 말하면 "532개의 과일이 있습니다."입니다. ㅋㅋ
현재 메시지 개수: 15
=============================== [Human Message] =================================

좋다. ㅋㅋ
현재 메시지 개수: 16
=============================== [Ai Message] ==================================

감사합니다! ㅋㅋ 더 필요하신 것이 있으면 말씀해 주세요!
현재 메시지 개수: 17
챗봇을 종료합니다.



**<<<<<<<<<<<<<<<<<< 기존 MEMORY 사용하여 다시 챗봇 접속했을때 >>>>>>>>>>>>>>>>>>**

대화형 챗봇을 시작합니다. 종료하려면 exit 또는 quit 를 입력하세요.
================================== Ai Message ==================================

감사합니다! ㅋㅋ 더 필요하신 것이 있으면 말씀해 주세요!
현재 메시지 개수: 17
================================ Human Message =================================

아까 뭐했지?
현재 메시지 개수: 18
================================== Ai Message ==================================

우리는 사과 317개와 배 215개의 총합이 532개란 것을 계산했어요. 그 후, 그 결과를 거꾸로 말해달라고 하셔서, 결과를 두 번 거꾸로 표현하는 대화도 나누었습니다.
현재 메시지 개수: 19
================================== Ai Message ==================================

우리는 사과 317개와 배 215개의 총합이 532개란 것을 계산했어요. 그 후, 그 결과를 거꾸로 말해달라고 하셔서, 결과를 두 번 거꾸로 표현하는 대화도 나누었습니다.
현재 메시지 개수: 19
================================ Human Message =================================

ㅂ
현재 메시지 개수: 20
================================== Ai Message ==================================

무엇을 도와드릴까요? 궁금한 점이 있으면 말씀해 주세요!
현재 메시지 개수: 21
챗봇을 종료합니다.
